# Seed Analysis: Inter-Seed Variability

Characterises how training seeds differ from each other across five eval domains.
The correlation analysis at the end identifies which differences co-vary.

| Domain | Eval(s) | What it measures |
|--------|---------|------------------|
| Training | `train_metrics.csv` | Reward convergence, food reward |
| Main eval | `m1a1k1_patchy_square` | Foraging, EOD, size → food slope |
| Sensor ablation (multi-fish) | `m?a?k?_patchy_square` | Collective foraging, biting, inequality by sensor condition |
| Sensor ablation (1-fish) | `1fish_m?a?k?_patchy_square` | Solo foraging by sensor condition |
| N-fish scaling | `nfish{1-4}_m1a1k1_patchy_square` | Group-size effect on foraging |
| 2f1p competition | `2f1p_{AltB,AeqB,AgtB,control_*}` | Size-dependent competitive outcomes |

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
import utils_seeds as us

# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
RESULTS_DIR       = Path("/home/satsingh/cluster_lab/satsingh/marl_fish_storage/results")
GROUP_FOLDER_NAME = "ConsNoise20260608dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU"
MAIN_EVAL         = "m1a1k1_patchy_square"
LAST_N            = 5
# ─────────────────────────────────────────────────────────────────────────────

run_dirs = us.discover_run_dirs(RESULTS_DIR, GROUP_FOLDER_NAME)
print(f"Group : {GROUP_FOLDER_NAME}")
print(f"Runs  : {len(run_dirs)}")

In [ ]:
df_all = us.load_analysis_df(run_dirs, main_eval=MAIN_EVAL, last_n=LAST_N)
print(f"Loaded {len(df_all)} seeds × {len(df_all.columns)} metrics.")

## Per-seed summary table

In [ ]:
show = [
    "seed",
    "train_final_reward", "train_final_r_food", "train_reward_std_tail",
    "food_mean", "food_std", "theil_mean", "biting_mean", "p_emit_eod",
    "size_food_slope", "size_food_r2", "size_food_p",
    "size_eod_slope", "size_disp_slope",
    "multi_sensing_benefit", "multi_morm_contrib", "multi_amp_contrib", "multi_knollen_contrib",
    "1fish_morm_contrib", "1fish_amp_contrib", "1fish_knollen_contrib",
    "food_nfish1", "food_nfish4", "nfish_slope",
    "b_food_size_effect", "b_bites_size_effect",
]
show = [c for c in show if c in df_all.columns]
df_key = df_all[show].copy()
float_cols = df_key.select_dtypes("float").columns.tolist()
grad_cols  = [c for c in ["size_food_slope", "food_mean", "size_food_r2", "multi_sensing_benefit"] if c in df_key.columns]
display(
    df_key.style
    .background_gradient(cmap="RdYlGn", subset=grad_cols, axis=0)
    .format({c: "{:.3f}" for c in float_cols})
)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
us.plot_training_curves(run_dirs, axes, last_n=LAST_N)
plt.tight_layout()
plt.show()

## Main eval: foraging and size-dependent behaviour

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
us.plot_main_eval_bars(df_all, axes)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
us.plot_main_eval_violins(run_dirs, MAIN_EVAL, axes)
plt.tight_layout()
plt.show()

## Sensor ablation: multi-fish (4-agent, patchy)

Same 4-agent setup as training. Captures collective effects of each sensor type.
Condition key: `m`=mormyromast, `a`=ampullary, `k`=knollen; 1=present, 0=absent.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
us.plot_sensor_multi_profiles(df_all, axes)
plt.suptitle("Multi-fish sensor ablation profiles (each line = one seed)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
us.plot_sensor_multi_contrib(df_all, axes)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
us.plot_sensor_multi_biting_theil(df_all, axes)
plt.tight_layout()
plt.show()

## Sensor ablation: single-fish

Individual foraging with each sensor removed — no social context.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
us.plot_sensor_1fish(df_all, axes)
plt.tight_layout()
plt.show()

## N-fish scaling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
us.plot_nfish(df_all, axes)
plt.tight_layout()
plt.show()

## 2f1p competitive outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
us.plot_2f1p(df_all, axes)
plt.tight_layout()
plt.show()

## Cross-metric correlation

Pearson correlations across seeds for all numeric metrics.
Reveals which inter-seed differences co-vary.

In [ ]:
g, corr = us.plot_corr_clustermap(df_all)
if g is not None:
    plt.show()

In [ ]:
if corr is not None:
    display(us.top_correlations(corr, n=25))

In [ ]:
if len(df_all) >= 3:
    g2 = us.plot_scatter_matrix(df_all)
    plt.show()